# VAE training and processing

Code to train a new VAE and run the CSI processing.

## Setup

In [1]:
import csv
import zipfile
from datetime import UTC
from datetime import datetime as dt
from pathlib import Path
from string import ascii_uppercase
from urllib.request import urlretrieve

import numpy as np
import scipy.io as sio
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Dataset config
DATASET_URL = "https://zenodo.org/record/7732595/files/S1.zip"
"""Zenodo URL for the S1 dataset."""
DATASET_PATH = Path("dataset/S1")
"""Local path to store the S1 dataset."""
N_ACTIVITIES = 12
"""Total number of activities in the dataset"""
N_SAMPLES = 12000
"""Number of samples to extract from each CSI matrix file."""
WINDOW_SIZE = 450
"""Size of the sliding window to extract from each sample."""
N_ANTENNAS = 1
"""Total number of antennas used, either a single one or all of them."""
ANTENNA = 0
"""If N_ANTENNAS==1, select which antenna to use (0 to 3). Otherwise, this value is ignored."""

# Categorical VAE config
LATENT_DIM = 2
CATEGORICAL_DIM = N_ACTIVITIES
VAE_NAME = f"vaed_s1a_a{ANTENNA}_ls{LATENT_DIM}" if N_ANTENNAS == 1 else f"vaed_s1a_f_ls{LATENT_DIM}"
CHECKPOINT_DIR = Path(f"./vaed_models_{N_ACTIVITIES}activities/{dt.now(tz=UTC).strftime('%Y%m%d_%H%M%S')}/{VAE_NAME}")

# Training config
BATCH_SIZE = 25
NUM_EPOCHS = 1
PATIENCE = 3
LEARNING_RATE = 1e-3
NUM_GPUS = torch.cuda.device_count()
USE_GPU = NUM_GPUS > 0
MODEL_DEVICE = torch.device("cuda" if USE_GPU else "cpu")

### Distributed Data Parallel (DDP) configuration

In [2]:
import os

from torch.distributed import init_process_group


def ddp_setup(rank: int, world_size: int) -> None:
    """Initialize the distributed environment.

    Arguments:
        rank: Unique identifier of each process
        world_size: Total number of processes

    """
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"

    # Set default GPU for the process
    torch.cuda.set_device(rank)
    init_process_group(backend="nccl", rank=rank, world_size=world_size)


### Download the dataset

In [3]:
if not DATASET_PATH.exists():
    DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)

    zip_file_path = DATASET_PATH.with_suffix(".zip")

    urlretrieve(DATASET_URL, zip_file_path)

    with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
        zip_ref.extractall("dataset")

    zip_file_path.unlink()

## Dataset

In [4]:
class CSIDataset(Dataset):
    """CSI Dataset for PyTorch.

    Shape of dataset items is (n_antennas, window_size, n_subcarriers)
    """

    def __init__(
        self,
        files: list[Path],
        n_samples: int,
        window_size: int,
        n_antennas: int,
        antenna_select: int,
        normalize: bool = True,
    ):
        self.window_size = window_size
        self.n_antennas = n_antennas
        self.normalize = normalize

        self.data = []
        self.labels = []
        self.index_map = []

        global_max = 0.0

        # Load files once, build index map
        for label, file in enumerate(files):
            # num_samples, n_subcarriers, n_antennas
            mat = sio.loadmat(file)

            csi = np.array(mat["csi"])
            csi = csi[:n_samples, ..., int(antenna_select)] if n_antennas == 1 else csi[:n_samples]
            csi = np.round(np.abs(csi)).astype(np.float32)

            if normalize:
                global_max = max(global_max, csi.max())

            file_id = len(self.data)
            self.data.append(csi)
            self.labels.append(label)

            # Build lazy sliding-window index
            for start in range(n_samples - window_size):
                self.index_map.append((file_id, start))

        self.global_max = global_max if normalize else 1.0

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx: int):
        file_id, start = self.index_map[idx]
        csi = self.data[file_id]

        window = csi[start : start + self.window_size]

        # (window_size, n_subcarriers, n_antennas) → (n_antennas, window_size, n_subcarriers)
        # The window_size represents the time dimension
        window = window[np.newaxis, ...] if self.n_antennas == 1 else np.transpose(window, (2, 0, 1))

        x = torch.from_numpy(window) / self.global_max
        y = self.labels[file_id]

        return x, y

In [5]:
files = [DATASET_PATH / f"S1a_{x}.mat" for x in ascii_uppercase[:N_ACTIVITIES]]
print([f.name for f in files])

# Shape of dataset samples: (n_antennas, window_size, n_subcarriers)
dataset = CSIDataset(
    files=files,
    n_samples=N_SAMPLES,
    window_size=WINDOW_SIZE,
    n_antennas=N_ANTENNAS,
    antenna_select=ANTENNA,
)

# Shape of dataloader batches: (batch_size, n_antennas, window_size, n_subcarriers)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=USE_GPU,
)

['S1a_A.mat', 'S1a_B.mat', 'S1a_C.mat', 'S1a_D.mat', 'S1a_E.mat', 'S1a_F.mat', 'S1a_G.mat', 'S1a_H.mat', 'S1a_I.mat', 'S1a_J.mat', 'S1a_K.mat', 'S1a_L.mat']


## Variational Auto-Encoder

### Gumbel Sampling

In [6]:
@torch.no_grad()
def _sample_gumbel_like(tensor: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Draw Gumbel(0,1) noise of the same shape as `tensor` with robust clamping of uniforms.

    Args:
        tensor (torch.Tensor): Tensor whose shape to match.
        eps (float): Small constant for numerical stability.

    Returns:
        torch.Tensor: Sampled Gumbel noise.

    """
    # Clamp away from {0,1} to avoid -log(-log(U)) overflow/NaN
    u = torch.rand_like(tensor).clamp_(eps, 1.0 - eps)
    return -torch.log(-torch.log(u))


def gumbel_softmax_stable(
    logits: torch.Tensor,
    tau: float,
    dim: int = -1,
    eps_u: float = 1e-6,
    center: bool = True,
) -> torch.Tensor:
    """Numerically stable (soft) Gumbel-Softmax.

    Clamps uniforms, enforces a minimum temperature,
    optional centering before softmax (shift-invariant, avoids overflow).

    Args:
        logits (torch.Tensor): Logits of the categorical distribution.
        tau (float): Temperature parameter.
        dim (int): Dimension along which to apply softmax.
        eps_u (float): Small constant for numerical stability of uniform sampling.
        center (bool): Whether to center logits before softmax.

    Returns:
        torch.Tensor: Sampled tensor from the Gumbel-Softmax distribution.

    """
    tau_eff = max(float(tau), 1e-3)
    g = _sample_gumbel_like(logits, eps=eps_u)
    y = logits + g
    if center:
        y = y - y.max(dim=dim, keepdim=True).values
    y = y / tau_eff
    return nn.functional.softmax(y, dim=dim)


def gumbel_softmax_straight_through_stable(
    logits: torch.Tensor,
    tau: float,
    dim: int = -1,
    eps_u: float = 1e-6,
    center: bool = True,
) -> torch.Tensor:
    """Straight-Through Gumbel-Softmax.

    In forward pass we get a hard one-hot sample, in backward gradients flow as if soft (y_soft).

    Args:
        logits (torch.Tensor): Logits of the categorical distribution.
        tau (float): Temperature parameter.
        dim (int): Dimension along which to apply softmax.
        eps_u (float): Small constant for numerical stability of uniform sampling.
        center (bool): Whether to center logits before softmax.

    Returns:
        torch.Tensor: Sampled tensor from the Gumbel-Softmax distribution with straight-through estimator.

    """
    y_soft = gumbel_softmax_stable(logits, tau=tau, dim=dim, eps_u=eps_u, center=center)

    # Hard one-hot in forward
    _, k = y_soft.max(dim=dim, keepdim=True)
    y_hard = torch.zeros_like(y_soft).scatter_(dim, k, 1.0)

    # Straight-through trick: forward = y_hard, backward = y_soft
    return (y_hard - y_soft).detach() + y_soft

### Encoder

In [7]:
class CSIEncoder(nn.Module):
    """CSI encoder module for VAE."""

    def __init__(self, input_shape: tuple[int, int, int], latent_dim: int, categorical_dim: int):
        super().__init__()

        self.latent_dim = latent_dim
        self.categorical_dim = categorical_dim

        self.conv = nn.Sequential(
            nn.Conv2d(input_shape[2], 32, kernel_size=(5, 8), stride=(5, 8)),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=(5, 8), stride=(5, 8)),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=(2, 4), stride=(2, 4)),
            nn.ReLU(),
        )

        # Infer flattened size dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, input_shape[2], input_shape[0], input_shape[1])
            flat_dim = self.conv(dummy).view(1, -1).size(1)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 24),
            nn.ReLU(),
        )

    def forward(
        self,
        x: torch.Tensor,
        tau: float,
        eps_u: float = 1e-6,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Forward pass through the encoder."""
        z = self.conv(x)
        z = self.fc(z)
        z = z.view(-1, self.latent_dim, self.categorical_dim)

        # Ensure float and pre-center logits to reduce magnitude (safe due to softmax shift-invariance)
        z = z.float()
        z = z - z.detach().amax(dim=-1, keepdim=True)

        # Straight-Through Gumbel-Softmax: hard forward, soft gradient
        z_gumbel = gumbel_softmax_straight_through_stable(
            z,
            tau=tau,
            dim=-1,
            eps_u=eps_u,
            center=False,  # already centered above
        )

        # Flatten [B, D, K] -> [B, D*K] for the latent vector
        z = z_gumbel.view(-1, self.latent_dim * self.categorical_dim)
        return z, z_gumbel

### Decoder

In [8]:
class CSIDecoder(nn.Module):
    """CSI decoder module for VAE."""

    def __init__(self, input_shape: tuple[int, int, int], latent_dim: int, categorical_dim: int, out_filter: int):
        super().__init__()

        self.input_shape = input_shape
        flat_dim = input_shape[0] * input_shape[1] * input_shape[2]

        self.fc = nn.Sequential(
            nn.Linear(latent_dim * categorical_dim, flat_dim),
            nn.ReLU(),
        )

        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(input_shape[2], 32, kernel_size=(2, 4), stride=(2, 4), padding=0),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 32, kernel_size=(5, 8), stride=(5, 8), padding=2, output_padding=(4, 4)),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 32, kernel_size=(5, 8), stride=(5, 8), padding=2, output_padding=(4, 4)),
            nn.ReLU(),
            nn.ConvTranspose2d(32, out_filter, kernel_size=out_filter),
            nn.Sigmoid(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """Forward pass through the decoder."""
        x = self.fc(z)
        x = x.view(
            z.size(0),
            self.input_shape[2],
            self.input_shape[0],
            self.input_shape[1],
        )
        return self.deconv(x)

### Full VAE model

In [9]:
from typing import Literal


class VAE(nn.Module):
    """Variational Autoencoder for CSI data."""

    def __init__(
        self,
        enc_input_shape: tuple[int, int, int] = (450, 2048, 1),
        dec_input_shape: tuple[int, int, int] = (9, 8, 32),
        latent_dim: int = 2,
        categorical_dim: int = 2,
    ) -> None:
        super().__init__()

        self.encoder = CSIEncoder(enc_input_shape, latent_dim, categorical_dim)
        self.decoder = CSIDecoder(dec_input_shape, latent_dim, categorical_dim, enc_input_shape[-1])

    def forward(
        self,
        x: torch.Tensor,
        tau: float,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Forward pass through the VAE."""
        z, _ = self.encoder(x, tau)
        return self.decoder(z), z

### Loss function

In [10]:
def vae_loss(
    x_recon: torch.Tensor,
    x_true: torch.Tensor,
    z: torch.Tensor,
    prior_prob: float,
    kl_weight: float = 5e-4,
    free_bits: float = 0.0,
    entropy_weight: float = 0.0,
    entropy_mode: Literal["none", "penalty", "bonus"] = "none",
    eps: float = 1e-12,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute the VAE loss with categorical latent variables.

    Args:
        x_recon (torch.Tensor): Reconstructed input.
        x_true (torch.Tensor): True input.
        z (torch.Tensor): Latent variable tensor.
        kl_weight (float): Weight for the KL divergence term.
        free_bits (float): Free bits threshold for KL divergence.
        prior_prob (float): Prior probability for the categorical distribution.
        entropy_weight (float): Weight for the entropy term.
        entropy_mode (Literal["none", "penalty", "bonus"]): Mode for entropy term.
        eps (float): Small constant for numerical stability.

    Returns:
        tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]: Total loss, reconstruction loss,
            KL divergence, entropy, entropy term.

    """
    # Reconstruction loss
    recon = nn.functional.binary_cross_entropy(x_recon, x_true, reduction="mean")

    # Log q(y|x)
    log_z = nn.functional.log_softmax(z, dim=-1)
    log_z = log_z.clamp_min(eps)

    # Posterior q(y|x)
    probs_z = nn.functional.softmax(z, dim=-1)
    posterior_distrib = torch.distributions.Categorical(probs=probs_z)
    posterior = posterior_distrib.probs

    # Use log_softmax for stability, then build q from it
    # probs_z = nn.functional.log_softmax(z, dim=-1)  # can be -inf in extreme cases
    # probs_z = probs_z.exp()  # q in [0,1], but can be exact 0
    # probs_z = probs_z.clamp_min(eps)  # avoid 0 * (-inf)
    # posterior_distrib = torch.distributions.Categorical(probs=probs_z)
    # posterior = posterior_distrib.probs
    # log_z = probs_z.log()

    # Prior p(y): uniform categorical
    prior = torch.ones_like(z) * prior_prob
    prior = prior.clamp_min(eps)
    prior_distrib = torch.distributions.Categorical(probs=prior)
    log_prior = prior_distrib.probs.log()

    # KL(q || p)
    kl_per_sample = (posterior * (log_z - log_prior)).view(z.size(0), -1).sum(dim=1)
    if free_bits > 0.0:
        kl_per_sample = nn.functional.relu(kl_per_sample - free_bits)
    kl = kl_per_sample.mean()

    # Entropy H(q): -sum q log q, safe version
    entropy_per_sample = -(posterior * log_z).view(z.size(0), -1).sum(dim=1)
    entropy = entropy_per_sample.mean()

    # Entropy term (sign depends on mode)
    if entropy_mode == "penalty":
        ent_term = +entropy_weight * entropy
    elif entropy_mode == "bonus":
        ent_term = -entropy_weight * entropy
    else:
        ent_term = torch.Tensor([0.0]).to(x_recon.device)

    total_loss = recon + kl_weight * kl + ent_term

    return total_loss, recon, kl, entropy, ent_term

## Training

In [11]:
from safetensors.torch import save_file

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

log_path = f"{CHECKPOINT_DIR}/model_history_log.csv"
file_exists = Path(log_path).is_file()

with Path(log_path).open("a", newline="") as csv_file:
    csv_writer = csv.writer(csv_file)
    if not file_exists:
        csv_writer.writerow(["epoch", "loss", "reconstruction_loss", "kl_loss"])

def save_checkpoint(model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int) -> None:
    """Save model and optimizer state dicts as a checkpoint."""
    path = CHECKPOINT_DIR / f"cp-{epoch:04d}.safetensors"
    state_dict = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    save_file(state_dict, path)


class EarlyStopping:
    """Early stopping utility to halt training when validation loss stops improving."""

    def __init__(self, patience: int) -> None:
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0

    def step(self, loss: float) -> bool:
        """Check if training should stop early based on loss improvement."""
        # If loss improved, reset counter and continue training
        if loss < self.best_loss:
            self.best_loss = loss
            self.counter = 0
            return False

        # If loss did not improve, increment counter
        self.counter += 1
        return self.counter >= self.patience

In [13]:
from collections import OrderedDict

from tqdm import tqdm

model = VAE(latent_dim=LATENT_DIM, categorical_dim=CATEGORICAL_DIM).to(MODEL_DEVICE)
model.train()

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
early_stopping = EarlyStopping(patience=PATIENCE)

for epoch in range(NUM_EPOCHS):
    model.train()

    epoch_loss = 0.0
    epoch_recon = 0.0
    epoch_kl = 0.0

    with tqdm(dataloader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}", unit="batch") as progress_bar:
        for x, _ in progress_bar:
            x_true = x.to(MODEL_DEVICE)

            optimizer.zero_grad()
            x_recon, z = model(x_true, 0.1)
            loss, recon_loss, kl_loss, _, _ = vae_loss(x_recon, x_true, z, prior_prob=1.0 / CATEGORICAL_DIM)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_recon += recon_loss.item()
            epoch_kl += kl_loss.item()

            progress_bar.set_postfix(
                OrderedDict(
                    [
                        ("loss", loss.item()),
                        ("recon_loss", recon_loss.item()),
                        ("kl_loss", kl_loss.item()),
                    ],
                ),
            )

    epoch_loss /= len(dataloader)
    epoch_recon /= len(dataloader)
    epoch_kl /= len(dataloader)

    save_checkpoint(model, optimizer, epoch)
    csv_writer.writerow([epoch, epoch_loss, epoch_recon, epoch_kl])
    csv_file.flush()

    if early_stopping.step(epoch_loss):
        break

Epoch 1/1:   0%|          | 10/5544 [00:15<2:22:33,  1.55s/batch, loss=0.658, recon_loss=0.657, kl_loss=3.18]


KeyboardInterrupt: 